In [ ]:
from collections import defaultdict
import random
import math

In [ ]:
# 1. Recycling Robot - MDP Defination

states = ["high", "low"]

actions = {"high": ["search", "wait"], "low": ["search", "wait", "recharge"]}

for state, available_action in actions.items():
    print(f"{state} -> {available_action}")

In [ ]:
# 2. Define the transition states
alpha = 0.8
beta = 0.7

P = {
    ("high", "search"): {"high": alpha, "low": 1 - alpha},
    ("high", "wait"): {"high": 1.0},
    ("low", "search"): {"low": beta, "high": 1 - beta},
    ("low", "wait"): {"low": 1.0},
    ("low", "recharge"): {"high": 1.0},
}

# Now check the fundamental probability rule:
for (state, action), transitions in P.items():
    print(state, action, sum(transitions.values()))

In [ ]:
# 3. Define the rewards r(s,a,s')
"""
r(s,a,s′) = expected reward for that particular transition or "immediate reward"
"""

r_search = 1.0
r_wait = 0.2

R = {
    ("high", "search", "high"): r_search,
    ("high", "search", "low"): r_search,

    ("high", "wait", "high"): r_wait,

    ("low", "search", "low"): r_search,
    ("low", "search", "high"): -3.0,

    ("low", "wait", "low"): r_wait,

    ("low", "recharge", "high"): 0
}


In [ ]:
def step(state, action):

    transitions = P[(state, action)]

    # Sample next state according to P
    next_states = list(transitions.keys())
    probabilities = list(transitions.values())

    next_state = random.choices(
        next_states,
        weights=probabilities
    )[0]

    # Get reward from R
    reward = R[(state, action, next_state)]

    return next_state, reward

In [ ]:
def take_action_randomly(state):
    action = random.choices(actions[state])[0]
    print(f"taking action: {action}")
    return action

In [ ]:
# run the process
trajectory_col = []
# set_list_of_actions = list(set([action for actions in actions.values() for action in actions]))
state = random.choice(states)
total_steps = 10

for i in range(total_steps):
    action = take_action_randomly(state)
    next_state, reward = step(state, action)
    trajectory_col.append((state, action, reward, next_state))
    print(f"S{i}:{state}, A{i}:{action}, R{i+1}:({reward}), S{i+1}:{next_state}")
    state = next_state
trajectory_col

In [ ]:
# dicount factor 
discount_factor = 0.9

# define return 
def calculate_returns(rewards, discount_factor):
    
    returns = []
    rewards_len = len(rewards)
    
    for t in range(rewards_len):
        G = 0
        
        for k in range(t, rewards_len):
            G += (discount_factor ** (k-t)) * rewards[k]
        
        returns.append(G)
    
    return returns

# test it 
rewards = [reward for state, action, reward, next_state in trajectory_col]
returns = calculate_returns(rewards, discount_factor)

for t, G in enumerate(returns):
    print(f"G_{t} = {G:.4f}")



In [ ]:
# Calculate the empirical value of each state
def calculate_epirical_state_value(trajectory_epi_data, returns):
    state_returns = []

    for t, (state, action, reward, next_state) in enumerate(trajectory_epi_data):
        state_returns.append((state, returns[t]))
    
    epirical_state_value = defaultdict(list)
    for state, G in state_returns:
        epirical_state_value[state].append(G)
    
    return dict(epirical_state_value)

In [ ]:
epirical_state_value = calculate_epirical_state_value(trajectory_col,returns)
epirical_state_value

In [ ]:
# state-value - V_pi : Vπ(s)
# V_pi = {}
# for state, state_returns in returns_by_value.items():
#     V_pi[state] = sum(state_returns) / len(state_returns)

# V_pi

def calculate_V_pi(empirical_state_value_data):
    V_pi = defaultdict(float)

    for state in states:
        returns = empirical_state_value_data.get(state, [])

        if returns:
            V_pi[state] = sum(returns) / len(returns)
        else:
            V_pi[state] = 0.0

    return dict(V_pi)


In [ ]:
V_pi = calculate_V_pi(epirical_state_value)
V_pi

In [ ]:
# action-value - Q_Value : Qπ(s,a)


def Q_pi(state, action, v_pi):
    
    value = 0

    for next_state, probabilities in P[(state, action)].items():
        immediate_reward = R[(state, action, next_state)]
        value += probabilities * (immediate_reward + discount_factor * v_pi[next_state])
    return value

# Q_pi Test


In [ ]:
def each_action_Q_pi(state, v_pi):
    actions_Q_pi = defaultdict(float)
    for action in actions[state]:
        actions_Q_pi[action] = Q_pi(state=state, action=action, v_pi=v_pi)
    return dict(actions_Q_pi)

# Test each_action_Q_pi
each_action_Q_pi("low", V_pi)

In [ ]:
def convert_each_action_q_pi_to_policy(q_values_dict, temperature=1.0):

    exp_values = {
        action: math.exp(q / temperature)
        for action, q in q_values_dict.items()
    }

    total = sum(exp_values.values())

    return {
        action: value / total
        for action, value in exp_values.items()
    }

# Test convert_each_action_q_pi_to_policy
convert_each_action_q_pi_to_policy(each_action_Q_pi("high", V_pi))



In [ ]:
def bellman_expectation(state, V_pi):
    value = 0

    for action, action_probability in stocastic_policy[state].items():

        q = Q_pi(state, action, V_pi)

        value += action_probability * q

    return value

bellman_expectation("high", V_pi)

In [ ]:
# redefine new policy
stocastic_policy = {
    "high": {"search": 1/2, "wait": 1/2},
    "low": {
        "search": 1/3,
        "wait": 1/3,
        "recharge": 1/3,
    },
}

In [ ]:
def take_action_based_on_stocastic_policy(state):
    next_possible_actions_probs = stocastic_policy[state]
    next_actions = list(next_possible_actions_probs.keys())
    probabilities = list(next_possible_actions_probs.values())
    
    next_action = random.choices(
        next_actions,
        weights=probabilities
    )[0]
    return next_action


In [ ]:
# Running whole experience to learning to update the "recycling robot"
episodes = 10
steps_within_epi = 100

discount_factor = 0.9

for epi in range(episodes):

    state = random.choice(states)

    trajectory_col = []
    
    print(f"\n--------\nbefore update :: {epi} -> {stocastic_policy=}")

    for t in range(steps_within_epi):

        # choose action
        action = take_action_based_on_stocastic_policy(state)

        # interact with env
        next_state, reward = step(state, action)

        # collect experience
        trajectory_col.append((state, action, reward, next_state))
        
        # next state 
        state = next_state

    # calculate trajectory_col -> rewards -> Gt -> V_pi -> Q_pi -> update stocastic_policy

    rewards = [reward for state, action, reward, next_state in trajectory_col]

    returns = calculate_returns(rewards=rewards, discount_factor=discount_factor)
    print(f"{returns=}")
    # return : list(G0, G1,... Gn) (discounted return from each state)

    from_each_state_return_col = calculate_epirical_state_value(
        trajectory_epi_data=trajectory_col, returns=returns
    )
    print(f"{from_each_state_return_col=}")
    # return : {'low': [0.043, ..., 0.0], 'high': [3.178541, 3.59, 1.0]}

    v_pi = calculate_V_pi(from_each_state_return_col)
    print(f"{v_pi=}")
    # return : {'low': 1.249, 'high': 2.94}

    # calculate each state -> possible actios Q-value -> softmax to prob -> update policy
    for state in states:

        each_possible_actions_Q_pi_values = each_action_Q_pi(state, v_pi)
        print(f"\n{each_possible_actions_Q_pi_values=}")
        # return : {'search': 1.179, 'wait': 1.92} if state = "high"

        update_state_policy_value = convert_each_action_q_pi_to_policy(
            each_possible_actions_Q_pi_values
        )
        print(f"{state} : {update_state_policy_value=}")
        # return: {'search': 0.64, 'wait': 0.36} if state = "high"

        # finally: update policy
        stocastic_policy[state] = update_state_policy_value
    
    print(f"\nafter update :: {epi} -> {stocastic_policy=}\n--------")